# Synthetic time-entry data: evaluation notebook (Stages 1-6)

**Goal.** Decide whether the synthetic corporate/transactional time-entry dataset (`synthetic_time_entries_v2.xlsx`, produced by `generate_synthetic_v2.py`) is good enough to build and evaluate a **matter-resolution pipeline**: given a submitted time entry, auto-match it to a matter, flag a mismatch, send it to review, or split it.

The evaluation follows the fidelity / utility / privacy framing used in the references (arXiv 2406.13130, the Medium and AWS posts on synthetic-data evaluation, and the IBM synthetic-data docs), adapted to a **template-generated** dataset with a known answer key.

https://arxiv.org/html/2406.13130v1

https://medium.com/data-science/evaluating-synthetic-data-the-million-dollar-question-a54701d1b621

https://dataplatform.cloud.ibm.com/docs/content/wsj/synthetic/evaluate_data_sd.html?context=wx

https://aws.amazon.com/blogs/machine-learning/how-to-evaluate-the-quality-of-the-synthetic-data-measuring-from-the-perspective-of-fidelity-utility-and-privacy/


| Stage | Question | Needs real data? |
|---|---|---|
| 1 Integrity | Is the answer key internally consistent? | No |
| 2 Split hygiene | Does TEST really measure generalization (no leakage, no accidental drift, enough rows)? | No |
| 3 Shortcuts | Can the labels be gamed without reading the narrative? | No |
| 0 Cleaning | Repair known parse defects in the real fee-application file | Real file |
| 4 Fidelity | Does the synthetic text look like real entries? | Real file |
| 5 Utility | Do models trained on synthetic data work on real data (TSTR vs TRTR)? | **Labeled** real sample |
| 6 Privacy / provenance | Was anything copied from real data? Where did the data come from? | Partly |

**Scorecard statuses:** `PASS` / `WARN` / `FAIL` against a gate, `INFO` (reported, no gate), `TODO` (manual step), `NOT RUN` (missing input).

**How to run:** set the paths in the config cell, then *Run All*. Every stage appends rows to one list (`RES`), which is shown per stage and again as a combined scorecard at the end.

## 0. Setup

Imports and configuration. Paths default to the current folder; set the `DATA_DIR` environment variable or edit the cells below. Set `REAL_CSV = None` to skip everything that needs the real fee-application file.

In [2]:
import os, re, hashlib, random, warnings
from collections import Counter
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

from scipy.sparse import csr_matrix, hstack
from scipy.spatial.distance import jensenshannon
from scipy.stats import chi2_contingency, ks_2samp, spearmanr, wasserstein_distance
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_colwidth", 120, "display.max_rows", 200)

In [3]:
DATA_DIR = "./data/"
SYNTHETIC_XLSX = os.path.join(DATA_DIR, "synthetic_time_entries_v3.xlsx")
GENERATOR_PY   = os.path.join(DATA_DIR, "generate_synthetic_v2.py")          # None to skip the provenance scan
REAL_CSV       = os.path.join(DATA_DIR, "time_entries_clean_balanced.csv")   # None to skip real-data stages
REAL_LABELS    = None   # CSV/XLSX with columns: narrative, quality (SPECIFIC / VAGUE / BLOCK_BILLED) -> enables Stage 5
OUT_DIR        = os.environ.get("OUT_DIR", "eval_outputs")
RUN_NEGATIVE_CONTROL = True   # Stage 4 sanity check: synthetic vs synthetic should be indistinguishable

if GENERATOR_PY and not os.path.exists(GENERATOR_PY): GENERATOR_PY = None
if REAL_CSV and not os.path.exists(REAL_CSV): REAL_CSV = None
os.makedirs(OUT_DIR, exist_ok=True)
print("synthetic :", os.path.basename(SYNTHETIC_XLSX))
print("generator :", os.path.basename(GENERATOR_PY) if GENERATOR_PY else "(skipped)")
print("real file :", os.path.basename(REAL_CSV) if REAL_CSV else "(skipped)")
print("labels    :", REAL_LABELS or "(none; Stage 5 will not run)")

synthetic : synthetic_time_entries_v3.xlsx
generator : (skipped)
real file : (skipped)
labels    : (none; Stage 5 will not run)


### Gates and helpers

Every check compares a measurement to a gate. **These thresholds are starting points to agree with stakeholders, not standards.** `add()` records a result; `st()` and `band()` turn a measurement into PASS/WARN/FAIL; `show()` renders a scorecard with colours.

In [4]:
MIN_TEST_PER_CLASS = 30
NN_GAP_MIN = 0.20          # TRAIN->TRAIN median similarity minus TEST->TRAIN median
NN_MAX_TEST = 0.90         # no TEST narrative this close to a TRAIN narrative
DRIFT_P_FAIL, DRIFT_P_WARN = 0.01, 0.10
HOURS_WASS_REL = 0.10      # Wasserstein as share of mean hours
SHORTCUT_MARGIN = 0.10     # shortcut macro-F1 may exceed majority by at most this
SHORTCUT_AUC_FAIL, SHORTCUT_AUC_WARN = 0.80, 0.70

NUM_W_PASS, NUM_W_WARN = 0.20, 0.50          # normalized Wasserstein (W / std of real)
BIN_D_PASS, BIN_D_WARN = 0.05, 0.15          # absolute difference in rate
DIST_AUC_PASS, DIST_AUC_WARN = 0.70, 0.85              # real-vs-synthetic distinguishability
MIN_N_STAGE4 = 100                           # per side
TSTR_PASS, TSTR_WARN = 0.85, 0.70            # TSTR macro-F1 as share of TRTR
NN_COPY_SIM = 0.95                           # synthetic narrative this close to a real one (>= 6 words) = copy

In [5]:
RES = []


def add(stage, check, value, gate, status, note=""):
    RES.append(dict(stage=stage, check=check, value=value, gate=gate, status=status, note=note))


def st(ok, warn=False):
    return "PASS" if ok else ("WARN" if warn else "FAIL")


def band(x, ok, warn):
    return "PASS" if x < ok else ("WARN" if x < warn else "FAIL")

In [6]:
def scorecard(prefix):
    """Rows recorded so far for the stage whose label starts with `prefix`."""
    df = pd.DataFrame(RES, columns=["stage", "check", "value", "gate", "status", "note"])
    return df[df.stage.str.startswith(prefix)]


def show(df, cols=("status", "check", "value", "gate")):
    colors = {"PASS": "#d4edda", "WARN": "#fff3cd", "FAIL": "#f8d7da", "INFO": "#e2e3e5",
              "NOT RUN": "#e2e3e5", "SKIPPED": "#e2e3e5", "TODO": "#cce5ff"}
    d = df[list(cols)].reset_index(drop=True)
    try:
        sty = d.style.apply(lambda r: [f"background-color: {colors.get(r['status'], '')}"] * len(r), axis=1)
        display(sty.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
    except Exception:          # Styler needs jinja2; fall back to a plain table
        display(d)

## 1. Load the synthetic workbook and look at it (EDA)

The workbook has four sheets: **Time Entries** (submitted fields plus gold `KEY:` answer-key columns), **TK Matter History** (each timekeeper's matters and 90-day hours), **Matter Master** (CMID to client/matter, codenames, aliases) and **Summary/Notes**. Columns starting `KEY:` are renamed `K_...`.

In [7]:
def load(path):
    te = pd.read_excel(path, "Time Entries", dtype={"CMID": str, "KEY: True CMID": str})
    mm = pd.read_excel(path, "Matter Master", dtype={"CMID": str})
    hist = pd.read_excel(path, "TK Matter History", dtype={"CMID": str})
    te.columns = [c.replace("KEY: ", "K_") for c in te.columns]
    try:
        summ = pd.read_excel(path, "Summary", header=None)
    except Exception:
        summ = None
    return te, mm, hist, summ


def norm_cmid(s, valid=None):
    if pd.isna(s):
        return None
    d = re.sub(r"\D", "", s)
    if len(d) == 7:
        return d[:4] + "." + d[4:]
    m = re.fullmatch(r"(\d{4})\.(\d{1,2})", s)
    if m:
        return m.group(1) + "." + m.group(2).zfill(3)
    m = re.fullmatch(r"(\d{4})[-/ ]\.?(\d{3})", s.strip())
    if m:
        return m.group(1) + "." + m.group(2)
    return None


def norm_text(s):
    return re.sub(r"[^a-z0-9 ]", "", str(s).lower()).strip()

In [8]:
te, mm, hist, summ = load(SYNTHETIC_XLSX)
print(f"{len(te)} time entries | {te['Time ID'].nunique()} unique IDs | {te['K_Split'].value_counts().to_dict()}")
print(f"{len(mm)} matters ({(mm.Status == 'Open').sum()} open) | {te['TK ID'].nunique()} timekeepers | dates {te['Date'].min()} to {te['Date'].max()}")
te.head(5)

4000 time entries | 4000 unique IDs | {'TRAIN': 2802, 'VAL': 601, 'TEST': 597}
103 matters (98 open) | 18 timekeepers | dates 07/06/2026 to 09/30/2026


,Time ID,Status,Source Format,TK ID,Timekeeper,Date,Entry Date,CMID,Client,Matter,...,K_Trivial,K_History Hint,K_Hard Case,K_CMID Form,K_Client Surface,K_Matter Surface,K_Narrative Quality,K_TK Style,K_Clean Narrative,K_Resolution Basis
0,T-00001,WIP,TYPED,TK101,A. Chen,07/06/2026,07/08/2026,7222.002,NaN,NaN,...,Y,NaN,NaN,EXACT,NONE,NONE,BLOCK_BILLED,terse_partner,"Correspondence with Kaur re specific indemnities for Summit Ridge (.6), turn comments on supplemental diligence requ...",Typed CMID consistent with entry
1,T-00002,WIP,PDF,TK101,A. Chen,07/06/2026,07/06/2026,NaN,NaN,NaN,...,N,NaN,NaN,NONE,NONE,NONE,SPECIFIC,terse_partner,Update time entries for week in preparation for closing.,task type => 1111.001
2,T-00003,WIP,TYPED,TK103,L. Moreno,07/06/2026,07/22/2026,5524.002,Cobalt Ridge Partners,NaN,...,Y,NaN,NaN,EXACT,CANONICAL,NONE,BLOCK_BILLED,lowercase_runon,Read through inv. register in preparation for signing (.5) + exchange emails w/ LP counsel re transfer of GP interes...,Typed CMID consistent with entry
3,T-00004,WIP,TYPED,TK103,L. Moreno,07/06/2026,07/09/2026,1111.001,Firm,Admin,...,Y,NaN,NaN,EXACT,CANONICAL,CANONICAL,SPECIFIC,lowercase_runon,Update time entries for week.,Typed CMID consistent with entry
4,T-00005,WIP,TYPED,TK104,D. Okafor,07/06/2026,07/06/2026,6103.1,NaN,NaN,...,N,Y,WRONG_CMID_SILENT,FORMAT,NONE,NONE,VAGUE,formal_full,Attend to filing.,Typed CMID (read '6103.1' as 6103.001) consistent with entry


**What to look for.** Class balance (the answer key's `Expected Action`), how the entered fields vary (`Input Type`), and whether TRAIN and TEST look alike. Rare classes (FLAG_MISMATCH, NEEDS_REVIEW, SPLIT_ENTRY) set how trustworthy the TEST metrics can be.

In [10]:
display(pd.crosstab(te["K_Expected Action"], te["K_Split"], margins=True))
display(pd.crosstab(te["K_Input Type"], te["K_Expected Action"], margins=True))

K_Split,TEST,TRAIN,VAL,All
K_Expected Action,,,,
AUTO_MATCH,452,2120,453,3025
FLAG_MISMATCH,36,168,36,240
NEEDS_REVIEW,36,166,38,240
NO_MATCH,36,174,37,247
SPLIT_ENTRY,37,174,37,248
All,597,2802,601,4000


KeyError: 'K_Input Type'

In [ ]:
te["_len"] = te["Narrative"].str.len()
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
pd.crosstab(te["K_Expected Action"], te["K_Split"]).plot.barh(ax=ax[0]); ax[0].set_title("Expected action by split")
te.boxplot(column="Hours", by="K_Narrative Quality", ax=ax[1]); ax[1].set_title("Hours by narrative quality"); ax[1].set_xlabel("")
te.boxplot(column="_len", by="K_Expected Action", ax=ax[2], rot=20); ax[2].set_title("Narrative length (chars) by action"); ax[2].set_xlabel("")
plt.suptitle(""); plt.tight_layout(); plt.show()
te = te.drop(columns="_len")

## 2. Stage 1: Integrity and label audit

**Goal:** confirm the answer key is internally consistent *before* using it to score anything.

Checks: unique IDs; every True CMID / client / matter agrees with the Matter Master; Input Type matches which fields are actually filled; entered client/matter agree with the entered CMID; AUTO_MATCH rows with a valid CMID resolve to it; Hard Case tags map to the right actions and relationships (closed matters, same-client wrong matter, off-history); flags are determinable; the Summary sheet counts match.

**Gate:** zero unexplained label conflicts. Documentation mismatches are WARN-level.

> Note: much of this passes *by construction*, because the generator writes the key and the fields together. Read it as a generator self-consistency test, not as evidence about real data.

In [ ]:
def stage1(te, mm, hist, summ):
    S = "1 Integrity"
    m = mm.set_index("CMID")
    valid = set(m.index)
    n = len(te)
    add(S, "Rows / unique Time IDs", f"{n} / {te['Time ID'].nunique()}", "equal", st(n == te["Time ID"].nunique()))

    single = te[~te["K_True CMID"].str.contains(r"\|")]
    bad = (~single["K_True CMID"].isin(valid)).sum()
    add(S, "True CMID exists in Matter Master", f"{bad} violations", "0", st(bad == 0))

    j = single.join(m[["Client", "Matter"]].add_prefix("M_"), on="K_True CMID")
    bad = ((j["K_True Client"] != j["M_Client"]) | (j["K_True Matter"] != j["M_Matter"])).sum()
    add(S, "True Client/Matter agree with master", f"{bad} violations", "0", st(bad == 0))

    # split entries use 'A | B' format, nothing else does
    sp = te["K_Expected Action"] == "SPLIT_ENTRY"
    bad = (te["K_True CMID"].str.contains(r"\|") != sp).sum()
    add(S, "'A | B' True CMID only on SPLIT_ENTRY", f"{bad} violations", "0", st(bad == 0))
    both_ok = te[sp]["K_True CMID"].map(
        lambda x: all(p.strip() in valid for p in x.split("|")) and len(x.split("|")) == 2
    ).all()
    add(S, "Both CMIDs in each split entry valid", str(both_ok), "True", st(bool(both_ok)))

    # Input Type vs filled fields
    fill = te.apply(lambda r: "".join("1" if pd.notna(r[c]) else "0" for c in ["CMID", "Client", "Matter"]), axis=1)
    expected = {"FULL": "111", "CMID_ONLY": "100", "CLIENT_ONLY": "010", "CODENAME": "001",
                "NONE": "000", "WRONG_CMID": "111", "INVALID_CMID": "100"}
    bad = (fill != te["K_Input Type"].map(expected)).sum()
    add(S, "Input Type matches fields actually filled", f"{bad} violations", "0", st(bad == 0))

    # entered client/matter consistent with entered CMID
    ncm = te["CMID"].map(norm_cmid)
    f = te[ncm.isin(valid) & te["Client"].notna()].copy()
    f["ncm"] = ncm[f.index]
    f = f.join(m[["Client", "Matter"]].add_prefix("M_"), on="ncm")
    bad = ((f["Client"] != f["M_Client"]) | (f["Matter"].notna() & (f["Matter"] != f["M_Matter"]))).sum()
    add(S, "Entered Client/Matter agree with entered CMID", f"{bad} violations", "0", st(bad == 0))

    # AUTO_MATCH with valid entered CMID resolves to it
    a = te[(te["K_Expected Action"] == "AUTO_MATCH") & ncm.isin(valid)]
    bad = (ncm[a.index] != a["K_True CMID"]).sum()
    add(S, "AUTO_MATCH with valid entered CMID: entered == True", f"{bad} of {len(a)} differ", "0", st(bad == 0))

    # hard-case <-> action mapping
    hc = te["K_Hard Case"].fillna("")
    act = te["K_Expected Action"]
    bad = (hc.str.contains("MULTI_MATTER") != (act == "SPLIT_ENTRY")).sum()
    add(S, "MULTI_MATTER <-> SPLIT_ENTRY", f"{bad} violations", "0", st(bad == 0))
    bad = (hc.str.contains("GENUINELY_AMBIGUOUS") != (act == "NEEDS_REVIEW")).sum()
    add(S, "GENUINELY_AMBIGUOUS <-> NEEDS_REVIEW", f"{bad} violations", "0", st(bad == 0))

    ent_status = ncm.map(lambda x: m.loc[x, "Status"] if x in valid else None)
    ent_client = ncm.map(lambda x: m.loc[x, "Client"] if x in valid else None)
    cm = hc.str.contains("CLOSED_MATTER")
    bad = (ent_status[cm] != "Closed").sum()
    add(S, "CLOSED_MATTER rows enter a closed CMID", f"{bad} of {cm.sum()} violate", "0", st(bad == 0))
    sw = hc.str.contains("SAME_CLIENT_WRONG_MATTER")
    bad = ((ent_status[sw] != "Open") | (ent_client[sw] != te.loc[sw, "K_True Client"])
           | (ncm[sw] == te.loc[sw, "K_True CMID"])).sum()
    add(S, "SAME_CLIENT_WRONG_MATTER: open, same client, different matter", f"{bad} of {sw.sum()} violate", "0", st(bad == 0))

    fl = te[act == "FLAG_MISMATCH"]
    bad = (ncm[fl.index] == fl["K_True CMID"]).sum()
    add(S, "FLAG_MISMATCH: entered CMID != True CMID", f"{bad} violations", "0", st(bad == 0))

    # OFF_HISTORY vs TK history
    pairs = set(zip(hist["TK ID"], hist["CMID"]))
    nomulti = te[~sp]
    inh = nomulti.apply(lambda r: (r["TK ID"], r["K_True CMID"]) in pairs, axis=1)
    off = nomulti["K_Hard Case"].fillna("").str.contains("OFF_HISTORY")
    bad = (inh == off).sum()
    add(S, "OFF_HISTORY <-> true matter not in TK history", f"{bad} violations", "0", st(bad == 0))

    # documentation consistency (WARN-level)
    nd = fl["K_Resolution Basis"].str.contains("not determinable", case=False).sum()
    add(S, "FLAG_MISMATCH rows with no determinable suggested matter", f"{nd} of {len(fl)}", "0 (Notes say all detectable)",
        st(nd == 0, warn=True), "Suggested-matter score is ill-defined for these rows; exclude or relabel.")
    untagged = fl["K_Hard Case"].isna().sum()
    add(S, "FLAG_MISMATCH rows with no Hard Case tag", f"{untagged} of {len(fl)}", "0",
        st(untagged == 0, warn=True), "Summary sheet undercounts SAME_CLIENT_WRONG_MATTER / adds an untagged wrong-client category.")

    fl_sc = fl[(ent_client[fl.index] == fl["K_True Client"]) & (ent_status[fl.index] == "Open")]
    mis = (~fl_sc["K_Hard Case"].fillna("").str.contains("SAME_CLIENT_WRONG_MATTER")).sum()
    add(S, "Same-client open wrong-matter flags missing SAME_CLIENT_WRONG_MATTER tag", f"{mis} of {len(fl_sc)}", "0",
        st(mis == 0), "Tag depends on which generator branch ran, not on the actual relationship; derive it after the fact.")

    # Summary sheet vs recomputed
    if summ is not None:
        def block(title):
            idx = summ.index[summ[0] == title]
            if len(idx) == 0:
                return None
            i = idx[0] + 1
            out = {}
            while i < len(summ) and pd.notna(summ.iloc[i, 0]) and summ.iloc[i, 0] != "Total":
                out[str(summ.iloc[i, 0])] = (summ.iloc[i, 1], summ.iloc[i, 2])
                i += 1
            return out
        bad, checked = 0, 0
        for title, col in [("Expected Action", "K_Expected Action"), ("Input Type", "K_Input Type"),
                           ("Narrative Quality", "K_Narrative Quality")]:
            b = block(title)
            if not b:
                continue
            ct = pd.crosstab(te[col], te["K_Split"])
            for lab, (tr, ts) in b.items():
                checked += 1
                try:
                    if int(ct.loc[lab, "TRAIN"]) != int(tr) or int(ct.loc[lab, "TEST"]) != int(ts):
                        bad += 1
                except Exception:
                    bad += 1
        add(S, "Summary sheet counts match recomputed", f"{bad} of {checked} rows differ", "0", st(bad == 0, warn=True),
            "Cached formula values; recalculate the workbook if this fails.")

In [ ]:
RES.clear()
stage1(te, mm, hist, summ)
show(scorecard("1"))

## 3. Stage 2: Split hygiene (leakage, drift, effective TEST size)

**Goal:** make sure TEST measures generalization rather than memorization.

- **Leakage:** exact narrative overlap; nearest-neighbour character similarity TEST to TRAIN versus TRAIN to TRAIN (TEST should be clearly less similar); reuse of TRAIN template IDs inside TEST rows; overlap of client misspellings.
- **Drift:** Jensen-Shannon distance and chi-square for categoricals, Wasserstein and KS for hours. Several columns are tested, so one borderline p-value is not alarming.
- **Effective size:** distinct narratives and templates in TEST, and rows per class (gate: at least 30 so one error moves recall by under about 3 points).

In [ ]:
def stage2(te, mm):
    S = "2 Split hygiene"
    te = te.copy()
    te["split"] = te["K_Split"]
    te["n"] = te["Narrative"].map(norm_text)
    tr, ts = te[te.split == "TRAIN"], te[te.split == "TEST"]

    ov = set(tr.n) & set(ts.n)
    add(S, "Exact narrative overlap TEST in TRAIN", f"{len(ov)} narratives", "0", st(len(ov) == 0))

    # nearest neighbour similarity
    trn = sorted(tr.n.unique())
    tsn = sorted(ts.n.unique())
    nn_test = np.array([max(SequenceMatcher(None, a, b).ratio() for b in trn) for a in tsn])
    nn_train = np.array([max(SequenceMatcher(None, a, b).ratio() for b in trn if b != a) for a in trn])
    gap = np.median(nn_train) - np.median(nn_test)
    add(S, "Nearest-neighbour char similarity (median)",
        f"TEST->TRAIN {np.median(nn_test):.2f} vs TRAIN->TRAIN {np.median(nn_train):.2f}",
        f"gap >= {NN_GAP_MIN}", st(gap >= NN_GAP_MIN))
    add(S, "Max TEST->TRAIN similarity", f"{nn_test.max():.2f}", f"< {NN_MAX_TEST}", st(nn_test.max() < NN_MAX_TEST))

    # template reuse
    ex = te.assign(t=te["K_Template IDs"].str.split("|")).explode("t")
    shared = set(ex[ex.split == "TRAIN"].t) & set(ex[ex.split == "TEST"].t)
    rows = ex[(ex.split == "TEST") & ex.t.isin(shared)].drop_duplicates("Time ID")
    acts = rows["K_Expected Action"].value_counts().to_dict()
    add(S, "TEST rows reusing a TRAIN template", f"{len(rows)} rows ({len(shared)} templates) {acts}", "0",
        st(len(rows) == 0, warn=True), "Whole narratives are new but components are not (typically split entries).")

    # misspelling overlap
    canon = set(mm["Client"]) | {x.strip() for s in mm["Known Aliases"].dropna() for x in s.split(",")}
    unl = te[te["K_Client Surface"] == "UNLISTED_VARIANT"]
    trv = set(unl[unl.split == "TRAIN"].Client)
    tsv = set(unl[unl.split == "TEST"].Client)
    add(S, "Unlisted client variants shared TRAIN/TEST", f"{len(trv & tsv)} (TRAIN {len(trv)}, TEST {len(tsv)})", "0",
        st(len(trv & tsv) == 0))
    leak = [v for v in (trv | tsv) if v in canon]
    add(S, "Unlisted variants found in Matter Master/aliases", f"{len(leak)}", "0", st(len(leak) == 0))

    # drift
    for col in ["K_Expected Action", "K_Input Type", "K_Narrative Quality", "K_Client Surface",
                "K_Hard Case", "TK ID", "K_True CMID"]:
        ct = pd.crosstab(te[col].fillna("NA"), te.split)
        p = chi2_contingency(ct)[1]
        a = ct["TRAIN"] / ct["TRAIN"].sum()
        b = ct["TEST"] / ct["TEST"].sum()
        jsd = jensenshannon(a.values, b.values, base=2)
        status = "FAIL" if p < DRIFT_P_FAIL else ("WARN" if p < DRIFT_P_WARN else "PASS")
        add(S, f"Drift TRAIN vs TEST: {col.replace('K_', '')}", f"JSD {jsd:.3f}, chi2 p {p:.3f}",
            f"p >= {DRIFT_P_WARN}", status,
            "Several columns are tested; treat single borderline p-values with caution." if status == "WARN" else "")
    w = wasserstein_distance(tr.Hours, ts.Hours)
    ksp = ks_2samp(tr.Hours, ts.Hours).pvalue
    add(S, "Drift TRAIN vs TEST: Hours", f"Wasserstein {w:.3f} ({w / te.Hours.mean():.1%} of mean), KS p {ksp:.2f}",
        f"< {HOURS_WASS_REL:.0%} of mean", st(w / te.Hours.mean() < HOURS_WASS_REL))

    # effective test size
    add(S, "TEST distinct narratives / template IDs", f"{ts.n.nunique()} / {ts['K_Template IDs'].nunique()} (of {len(ts)} rows)",
        "report", "INFO")
    for c, k in ts["K_Expected Action"].value_counts().items():
        add(S, f"TEST rows in class {c}", str(k), f">= {MIN_TEST_PER_CLASS}", st(k >= MIN_TEST_PER_CLASS),
            "One error moves recall by %.0f points." % (100 / k) if k < MIN_TEST_PER_CLASS else "")

In [ ]:
stage2(te, mm)
show(scorecard("2"))

## 4. Stage 3: Shortcut and difficulty audit

**Goal:** find out whether the benchmark can be gamed or is trivially easy.

- **Baselines** (trained on TRAIN, scored on TEST): majority class, a *shortcut* model that sees only hours / length / separator words, a model that sees only which fields were entered, a text model, and text plus entered fields plus hours. Reported as accuracy and **macro-F1 with a bootstrap 95% interval**, because the majority class makes accuracy misleading.
- **Trivial rule:** how many AUTO_MATCH rows are solved just by copying the entered CMID.
- **History as a prior:** how far "pick the timekeeper's highest-hours matter" gets.
- **Within-quality AUC:** a feature like length that merely separates multi-part from one-line entries is not a leak, so AUCs are computed *within* narrative-quality strata.
- **Class-exclusive tokens:** words that almost only occur in one class (for example split-only connectors).

In [ ]:
def boot_f1(y, p, labels, reps=300, seed=0):
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    vals = []
    for _ in range(reps):
        i = rng.integers(0, len(y), len(y))
        vals.append(f1_score(y[i], p[i], labels=labels, average="macro", zero_division=0))
    return np.percentile(vals, [2.5, 97.5])


def stage3(te, mm, hist):
    S = "3 Shortcuts"
    m = mm.set_index("CMID")
    valid = set(m.index)
    te = te.copy()
    te["ncm"] = te["CMID"].map(norm_cmid)
    te["f_cmid"] = te.CMID.notna().astype(int)
    te["f_client"] = te.Client.notna().astype(int)
    te["f_matter"] = te.Matter.notna().astype(int)
    te["cmid_valid"] = te.ncm.isin(valid).astype(int)
    te["cmid_closed"] = te.ncm.map(lambda x: int(x in valid and m.loc[x, "Status"] == "Closed"))
    te["nw"] = te.Narrative.str.split().str.len()
    te["len"] = te.Narrative.str.len()
    te["sep"] = te.Narrative.str.contains(r";|separately|also|\+", case=False).astype(int)

    y = te["K_Expected Action"]
    tr = (te.K_Split == "TRAIN").values
    ts = ~tr
    labels = sorted(y.unique())
    struct = ["f_cmid", "f_client", "f_matter", "cmid_valid", "cmid_closed"]
    short = ["Hours", "nw", "len", "sep"]

    def row(name, p, note="", status="INFO"):
        mf = f1_score(y[ts], p, labels=labels, average="macro", zero_division=0)
        lo, hi = boot_f1(y[ts], p, labels)
        add(S, f"Baseline {name}", f"acc {accuracy_score(y[ts], p):.3f}, macro-F1 {mf:.3f} [{lo:.2f}-{hi:.2f}]",
            "report" if status == "INFO" else f"<= majority + {SHORTCUT_MARGIN}", status, note)
        return mf

    maj = np.repeat(y[tr].mode()[0], ts.sum())
    mf_maj = row("majority class", maj)

    def gb(cols):
        return GradientBoostingClassifier(random_state=0).fit(te.loc[tr, cols], y[tr]).predict(te.loc[ts, cols])

    p = gb(short)
    mf = f1_score(y[ts], p, labels=labels, average="macro", zero_division=0)
    row("hours + length + separator words (no narrative content)", p,
        status=st(mf <= mf_maj + SHORTCUT_MARGIN), note="Pure shortcut model; should not beat majority by much.")
    row("entered-field presence only (legitimate signal)", gb(struct))

    tv = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True).fit(te.Narrative[tr])
    Xn = tv.transform(te.Narrative)
    Xf = hstack([Xn, csr_matrix(te[struct + ["Hours", "nw", "sep"]].values.astype(float))]).tocsr()
    p_txt = LogisticRegression(max_iter=3000, C=5, class_weight="balanced").fit(Xn[tr], y[tr]).predict(Xn[ts])
    row("narrative text only", p_txt)
    lr = LogisticRegression(max_iter=3000, C=5, class_weight="balanced").fit(Xf[tr], y[tr])
    p_all = lr.predict(Xf[ts])
    row("text + entered fields + hours", p_all)

    # trivial-rule share
    triv = (y == "AUTO_MATCH") & te.ncm.isin(valid) & (te.ncm == te["K_True CMID"])
    add(S, "AUTO_MATCH rows solved by copying the entered CMID",
        f"{triv.sum()} of {(y == 'AUTO_MATCH').sum()} AUTO_MATCH ({triv.sum() / len(te):.0%} of all; "
        f"{(triv & pd.Series(ts, index=te.index)).sum()} of {ts.sum()} in TEST)",
        "report; also score non-trivial subset", "INFO")
    nt = ts & ~triv.values
    mfn = f1_score(y[nt], p_all[~triv.values[ts]], labels=labels, average="macro", zero_division=0)
    add(S, "Best baseline on non-trivial TEST rows (action macro-F1)",
        f"{mfn:.3f} on {nt.sum()} rows (acc {accuracy_score(y[nt], p_all[~triv.values[ts]]):.3f})", "report", "INFO")

    # per-class recall of best baseline
    for c in labels:
        mask = (y[ts] == c).values
        rec = (p_all[mask] == c).mean() if mask.any() else float("nan")
        prec = (y[ts].values[p_all == c] == c).mean() if (p_all == c).any() else float("nan")
        add(S, f"Best baseline per-class: {c}", f"recall {rec:.2f}, precision {prec:.2f} (n={mask.sum()})", "report", "INFO")

    # history-as-prior baseline: how far does "pick the TK's highest-hours matter" get?
    hh = hist.merge(mm[["CMID", "Status"]], on="CMID")
    hh = hh[hh.Status == "Open"]
    sub = te[te["K_Input Type"].isin(["NONE", "CLIENT_ONLY"]) & ~te["K_True CMID"].str.contains(r"\|")]

    def top(r):
        c = hh[hh["TK ID"] == r["TK ID"]]
        if r["K_Input Type"] == "CLIENT_ONLY":
            c = c[c["Client"] == r["K_True Client"]]
        return c.loc[c["Hours (last 90 days)"].idxmax(), "CMID"] if len(c) else None

    acc = (sub.apply(top, axis=1) == sub["K_True CMID"]).mean()
    add(S, "History-as-prior baseline (highest-hours matter) on NONE+CLIENT_ONLY", f"{acc:.0%} (n={len(sub)})", "report", "INFO",
        "Shows how much of matter resolution the history table gives away. Compare with real-world reliance on history.")

    # stratified single-feature AUC: compare within Narrative Quality strata so a feature that merely
    # proxies "multi-part vs one-line" is not mistaken for a label leak
    qual = te["K_Narrative Quality"]

    def strat_auc(yb, x):
        num = den = 0.0
        for q in qual.unique():
            idx = (qual == q).values
            pos, neg = yb[idx].sum(), (1 - yb[idx]).sum()
            if pos >= 3 and neg >= 3:
                w = pos * neg
                num += w * roc_auc_score(yb[idx], x[idx])
                den += w
        return num / den if den else float("nan")

    for c in labels:
        yb = (y == c).astype(int).values
        for feat in ["Hours", "len", "nw"]:
            auc = strat_auc(yb, te[feat].values)
            if np.isnan(auc):
                continue
            dev = max(auc, 1 - auc)
            status = "FAIL" if dev >= SHORTCUT_AUC_FAIL else ("WARN" if dev >= SHORTCUT_AUC_WARN else "PASS")
            if status != "PASS" or feat == "Hours":
                add(S, f"Within-quality AUC: {feat} vs {c}", f"{auc:.2f}", f"within {1 - SHORTCUT_AUC_WARN:.2f}-{SHORTCUT_AUC_WARN:.2f}", status,
                    "Class is partly identifiable from this feature even among entries of the same narrative quality."
                    if status != "PASS" else "")

    # class-exclusive tokens
    from collections import Counter
    toks = te.Narrative.str.lower().str.findall(r"[a-z0-9&/+']+")
    doc = Counter()
    byc = {c: Counter() for c in labels}
    for t, c in zip(toks, y):
        for w in set(t):
            doc[w] += 1
            byc[c][w] += 1
    majority = y.value_counts().idxmax()
    for c in labels:
        if c == majority:
            continue
        tot = int((y == c).sum())
        hits = sorted([(w, byc[c][w], n) for w, n in doc.items()
                       if n >= 5 and byc[c][w] / n >= 0.90 and byc[c][w] / tot >= 0.25], key=lambda t: -t[1])
        if hits:
            add(S, f"Class-exclusive tokens for {c}", ", ".join(f"'{w}' {k}/{n}" for w, k, n in hits[:6]),
                "none (>=90% precision, >=25% recall)", "WARN",
                "Lexical marker that identifies the class without reading content; use it in other classes too or remove it.")

In [ ]:
stage3(te, mm, hist)
show(scorecard("3"))

### Scorecard so far (Stages 1-3)

In [ ]:
s13 = pd.DataFrame(RES)
display(s13.groupby("stage").status.value_counts().unstack(fill_value=0))
print("Items needing action:")
show(s13[s13.status.isin(["FAIL", "WARN"])])

## 5. Real data: cleaning the fee-application file

Stages 4-6 compare the synthetic data with **real billed narratives from bankruptcy fee applications**. The file was parsed from PDFs, and the parse has known defects, so it is repaired and audited first:

| Defect | Repair |
|---|---|
| `hours` taken from the first time slice instead of the entry total | recompute from the hours-like number in `raw_block` (verified below against the narrative's slice sum) |
| Table-footer text in the narrative ("TOTAL PHASE...", "Page 28", matter codes) | strip |
| Task label in the `timekeeper` field ("Examine Documents") with the initials stuck at the front of the narrative | move initials into `timekeeper` |
| Surname wrapped onto the end of the narrative ("... report. Lowenthal") | move to `timekeeper` |
| Mis-decoded UTF-8 ("Examinerâ€™s") | re-decode |
| `firm` prefixed with the filer ("X on behalf of Firm") | strip the prefix |

Rows whose hours cannot be verified (or whose slice sum contradicts the hours) are dropped rather than guessed. `high_confidence` is TRUE on every row, so it is not a usable filter.

In [ ]:
def read_any(path):
    return pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(path)


TASKISH = re.compile(r"^(examine|review|revision|draft|prepar|analy|confer|correspond|attend|research|telephon|email|"
                     r"meeting|document|internet|task summary|legal|general|case\b)", re.I)
FOOTERS = [r"\s*TOTAL\s+PHASE\b.*$", r"\s*Phase:\s*\S+.*$", r"\s*\bDATE\s+(?:TIMEKEEPER|ATTORNEY)\b.*$",
           r"\s*\bPage\s+\d+\b.*$", r"\s*\b\d{5,6}-\d{3,5}\b.*$"]


def fix_enc(s):
    if not isinstance(s, str):
        return s
    if "â€" in s or re.search(r"Ã[\u0080-\u00bf]", s):
        try:
            return s.encode("cp1252").decode("utf-8")
        except Exception:
            for a, b in [("â€™", "’"), ("â€œ", "“"), ("â€“", "–"), ("â€”", "—"), ("â€˜", "‘")]:
                s = s.replace(a, b)
    return s


def clean_real(df):
    """Repair the known parse defects; drop rows whose hours cannot be verified. Returns (clean_df, report_df)."""
    out, rep = [], []
    for i, r in df.iterrows():
        f = []
        g = lambda k: ("" if pd.isna(r.get(k)) else str(r.get(k)))
        nar0, raw, tk = g("narrative"), fix_enc(g("raw_block")), fix_enc(g("timekeeper"))
        nar = fix_enc(nar0)
        if nar != nar0:
            f.append("encoding_fixed")
        before = re.sub(r"\s+", " ", nar).strip()
        for p in FOOTERS:
            nar = re.sub(p, "", nar, flags=re.S)
        nar = re.sub(r"\s+", " ", nar).strip()
        if nar != before:
            f.append("footer_stripped")
        m = re.match(r"^([A-Z]{2,4})\s+(.*)$", nar)
        if TASKISH.match(tk) and m:
            tk, nar = m.group(1), m.group(2)
            f.append("tk_repaired")
        elif TASKISH.match(tk):
            f.append("tk_suspect")
        m2 = re.match(r"^(.*?[.!?])\s+([A-Z][a-z]{2,})$", nar)
        if re.fullmatch(r"[A-Z][a-z]+ [A-Z]\.", tk) and m2:
            tk, nar = f"{tk} {m2.group(2)}", m2.group(1)
            f.append("tk_surname_repaired")
        try:
            h = float(r.get("hours"))
        except Exception:
            h = np.nan
        nopar = re.sub(r"\([^)]*\)", "", raw, flags=re.S)
        toks = [float(t) for t in re.findall(r"(?<![\d,$.])\d*\.\d{1,2}(?!\d)", nopar)]
        hl = [t for t in toks if 0 < t <= 24]
        if np.isnan(h):
            f.append("hours_missing")
        elif toks:
            if not any(abs(t - h) < 0.005 for t in toks):
                if len(hl) == 1:
                    h = hl[0]
                    f.append("hours_repaired")
                else:
                    f.append("hours_unresolved")
        else:
            f.append("hours_unverified")
        sl = [float(x) for x in re.findall(r"\((\d*\.?\d+)\)", nar) if float(x) <= 24]
        if len(sl) >= 2 and not np.isnan(h) and abs(sum(sl) - h) > 0.05:
            f.append("slice_sum_mismatch")
        keep = len(nar) >= 3 and not ({"hours_unresolved", "hours_missing", "slice_sum_mismatch"} & set(f))
        rep.append(dict(row=i, kept=keep, repairs=";".join(f)))
        if keep:
            out.append(dict(date=r.get("date"), timekeeper=tk, hours=h, narrative=nar,
                            case_name=g("case_name"), firm=re.sub(r"^.*\bon behalf of\s+", "", g("firm")).strip(), repairs=";".join(f)))
    return pd.DataFrame(out), pd.DataFrame(rep)

In [ ]:
real, clean_rep = None, None
if REAL_CSV:
    raw_real = read_any(REAL_CSV)
    real, clean_rep = clean_real(raw_real)
    clean_rep.to_csv(os.path.join(OUT_DIR, "cleaning_report.csv"), index=False)
    fl = Counter(x for s_ in clean_rep["repairs"] for x in str(s_).split(";") if x)
    add("0 Real-data cleaning", "Rows in / kept", f"{len(raw_real)} / {len(real)}", "report", "INFO")
    for k, v in sorted(fl.items()):
        add("0 Real-data cleaning", f"Rows flagged: {k}", str(v), "report", "INFO")
    show(scorecard("0"))
else:
    print("No real file configured; Stages 4 and 6 will report NOT RUN.")

**Audit of the hours repair.** If the repair is right, the repaired hours should equal the sum of the parenthetical time slices in the narrative, and the original value should not.

In [ ]:
if REAL_CSV:
    rr = raw_real.loc[clean_rep.index[clean_rep["repairs"].str.contains("hours_repaired")]].copy()

    def new_hours(r):
        nopar = re.sub(r"\([^)]*\)", "", str(r.raw_block), flags=re.S)
        t = [float(x) for x in re.findall(r"(?<![\d,$.])\d*\.\d{1,2}(?!\d)", nopar) if 0 < float(x) <= 24]
        return t[0]

    rr["repaired"] = rr.apply(new_hours, axis=1)
    n = ok_old = ok_new = 0
    for _, r in rr.iterrows():
        sl = [float(x) for x in re.findall(r"\((\d*\.?\d+)\)", str(r.narrative)) if float(x) <= 24]
        if len(sl) >= 2:
            n += 1
            ok_old += abs(sum(sl) - r.hours) < 0.05
            ok_new += abs(sum(sl) - r.repaired) < 0.05
    print(f"{len(rr)} rows repaired ({len(rr) / len(raw_real):.1%} of file); firms: {rr.firm.str[:25].value_counts().head(3).to_dict()}")
    print(f"median repaired/original hours ratio: {(rr.repaired / rr.hours).median():.2f}")
    print(f"rows with >=2 time slices: {n} | slice sum matches ORIGINAL hours: {ok_old} | matches REPAIRED hours: {ok_new}")
    display(rr[["firm", "hours", "repaired", "narrative"]].head(5))

## 6. Stage 4: Fidelity against real entries

**Goal:** measure how realistic the synthetic entries look.

1. **Marginals:** length, casing, abbreviation rate, semicolons, digits, hours; and rates for lowercase starts, trailing periods, ALL-CAPS, time slices, block-billing and vague proxies (the same proxy definitions are applied to both sides). Numeric gap = Wasserstein distance divided by the real standard deviation.
2. **Joint structure:** Spearman correlation of word count with hours.
3. **Distinguishability:** a classifier tries to tell real from synthetic (5-fold CV AUC; 0.5 is indistinguishable). It is run on style features, on style plus hours, on a *shared vocabulary* that masks domain nouns, and on all words (which includes the domain gap, for reference). The top give-away features are listed.
4. **Blind review sheet:** 40 real and 40 synthetic entries shuffled for human raters, with a separate answer key.

> **Interpretation caveat.** The real data is *filed* fee-application text (edited, formal, often with per-task time slices). The generator imitates *raw* time entries with shorthand and typos. Part of any gap is therefore by design; what the test can still show is whether the generator's noise model is calibrated to anything observed.

In [ ]:
ABBR_TOK = {"tcw", "rvw", "revw", "cmts", "mgmt", "docs", "agmt", "agrmt", "drft", "prep", "conf", "dd", "w/", "t/c",
            "&", "+", "re", "mtg", "provs", "info", "discl."}


def style_frame(narr, hours=None):
    s = pd.Series(list(narr)).fillna("").astype(str)
    letters = s.str.count(r"[A-Za-z]")
    up = s.str.count(r"[A-Z]")
    f = pd.DataFrame({
        "chars": s.str.len(),
        "words": s.str.split().str.len(),
        "upper_share": (up / letters.replace(0, np.nan)).fillna(0),
        "starts_lower": s.str.match(r"^[a-z]").astype(int),
        "ends_period": s.str.contains(r"\.$").astype(int),
        "semicolons": s.str.count(";"),
        "paren_slices": s.str.count(r"\(\d*\.?\d+\)"),
        "digits": s.str.count(r"\d"),
        "abbr_rate": s.apply(lambda x: np.mean([w.lower().strip(".,;:") in ABBR_TOK for w in x.split()]) if x.split() else 0.0),
    })
    f["all_caps"] = (f.upper_share >= 0.9).astype(int)
    f["has_slices"] = (f.paren_slices > 0).astype(int)
    f["block_proxy"] = ((f.semicolons >= 1) | (f.paren_slices >= 2)).astype(int)
    f["vague_proxy"] = (f.words <= 4).astype(int)
    if hours is not None:
        h = pd.Series(list(hours)).astype(float)
        f["hours"] = h
        f["hours_tenth"] = np.isclose((h * 10).round(), h * 10, atol=1e-6).astype(int)
        f["hours_ge4"] = (h >= 4).astype(int)
    return f

In [ ]:
def cv_auc(X, y, model, seed=0):
    skf = StratifiedKFold(5, shuffle=True, random_state=seed)
    p = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
    return roc_auc_score(y, p)


def stage4(syn, real, out):
    S = "4 Fidelity"
    if real is None:
        add(S, "All Stage 4 checks", "NOT RUN", "needs real sample", "NOT RUN",
            "Provide --real (cleaned fee-application entries). Aim for several hundred rows or more, stratified by firm and case.")
        return
    nr, ns = len(real), len(syn)
    if min(nr, ns) < MIN_N_STAGE4:
        add(S, "Sample size", f"real {nr}, synthetic {ns}", f">= {MIN_N_STAGE4} each", "WARN",
            "Underpowered: distinguishability and marginal tests below are indicative only.")
    fr = style_frame(real.narrative, real.hours)
    fs = style_frame(syn.Narrative, syn.Hours)
    for col in ["chars", "words", "upper_share", "abbr_rate", "semicolons", "digits", "hours"]:
        w = wasserstein_distance(fr[col], fs[col])
        sd = fr[col].std() or 1e-9
        add(S, f"Marginal: {col}", f"real median {fr[col].median():.2f} vs synthetic {fs[col].median():.2f}; normalized W {w / sd:.2f}",
            f"< {NUM_W_PASS} (warn < {NUM_W_WARN})", band(w / sd, NUM_W_PASS, NUM_W_WARN))
    for col in ["starts_lower", "ends_period", "all_caps", "has_slices", "block_proxy", "vague_proxy", "hours_tenth", "hours_ge4"]:
        d = abs(fr[col].mean() - fs[col].mean())
        add(S, f"Rate: {col}", f"real {fr[col].mean():.2f} vs synthetic {fs[col].mean():.2f}",
            f"|diff| <= {BIN_D_PASS} (warn <= {BIN_D_WARN})", "PASS" if d <= BIN_D_PASS else ("WARN" if d <= BIN_D_WARN else "FAIL"),
            "Proxy definition, same on both sides." if col in ("block_proxy", "vague_proxy") else "")
    # joint structure: hours vs length
    cr = fr[["words", "hours"]].corr(method="spearman").iloc[0, 1]
    cs = fs[["words", "hours"]].corr(method="spearman").iloc[0, 1]
    add(S, "Joint: Spearman(words, hours)", f"real {cr:.2f} vs synthetic {cs:.2f}", "|diff| <= 0.15",
        "PASS" if abs(cr - cs) <= 0.15 else "WARN")

    # distinguishability
    n = min(nr, ns, 3000)
    rr = real.sample(n, random_state=0)
    ss = syn.sample(n, random_state=0)
    Xr, Xs = style_frame(rr.narrative, rr.hours), style_frame(ss.Narrative, ss.Hours)
    X = pd.concat([Xr, Xs], ignore_index=True)
    y = np.r_[np.ones(n), np.zeros(n)]
    nohrs = [c for c in X.columns if not c.startswith("hours")]
    gb = lambda: GradientBoostingClassifier(random_state=0)
    a1 = cv_auc(X[nohrs], y, gb())
    a2 = cv_auc(X, y, gb())
    imp = pd.Series(gb().fit(X, y).feature_importances_, index=X.columns).sort_values(ascending=False)
    add(S, "Distinguishability: style features (no hours)", f"AUC {a1:.2f}", f"< {DIST_AUC_PASS} (warn < {DIST_AUC_WARN})", band(a1, DIST_AUC_PASS, DIST_AUC_WARN))
    add(S, "Distinguishability: style + hours", f"AUC {a2:.2f}", f"< {DIST_AUC_PASS} (warn < {DIST_AUC_WARN})", band(a2, DIST_AUC_PASS, DIST_AUC_WARN),
        "Top give-aways: " + ", ".join(f"{k} ({v:.2f})" for k, v in imp.head(5).items()))
    texts = pd.concat([rr.narrative, ss.Narrative], ignore_index=True).astype(str)
    cr_, cs_ = Counter(), Counter()
    for t in rr.narrative.astype(str):
        cr_.update(set(re.findall(r"[a-z]+", t.lower())))
    for t in ss.Narrative.astype(str):
        cs_.update(set(re.findall(r"[a-z]+", t.lower())))
    vocab = sorted(w for w in cr_ if cr_[w] >= 3 and cs_.get(w, 0) >= 3)
    if len(vocab) >= 30:
        tv = TfidfVectorizer(vocabulary=vocab, lowercase=True, sublinear_tf=True)
        a3 = cv_auc(tv.fit_transform(texts), y, LogisticRegression(max_iter=2000))
        lr = LogisticRegression(max_iter=2000).fit(tv.transform(texts), y)
        co = pd.Series(lr.coef_[0], index=vocab).sort_values()
        add(S, "Distinguishability: shared-vocabulary words (domain nouns masked)", f"AUC {a3:.2f} ({len(vocab)} shared words)",
            f"< {DIST_AUC_PASS} (warn < {DIST_AUC_WARN})", band(a3, DIST_AUC_PASS, DIST_AUC_WARN),
            "Most real-leaning: " + ", ".join(co.index[-6:][::-1]) + ". Most synthetic-leaning: " + ", ".join(co.index[:6]))
    else:
        add(S, "Distinguishability: shared-vocabulary words", f"only {len(vocab)} shared words", ">= 30", "WARN",
            "Corpora barely overlap in vocabulary; domain gap dominates.")
    tv = TfidfVectorizer(sublinear_tf=True, min_df=2)
    a4 = cv_auc(tv.fit_transform(texts), y, LogisticRegression(max_iter=2000))
    add(S, "Distinguishability: all words (includes domain gap)", f"AUC {a4:.2f}", "report", "INFO",
        "Expected to be high across domains; compare with the masked result above.")

    # blind review sheet
    k = min(40, len(rr), len(ss))
    if k >= 10:
        a = pd.DataFrame({"narrative": rr.narrative.head(k).values, "hours": rr.hours.head(k).values, "source": "real"})
        b = pd.DataFrame({"narrative": ss.Narrative.head(k).values, "hours": ss.Hours.head(k).values, "source": "synthetic"})
        m = pd.concat([a, b]).sample(frac=1, random_state=1).reset_index(drop=True)
        m.insert(0, "id", range(1, len(m) + 1))
        m[["id", "narrative", "hours"]].assign(rating_looks_real_1to5="").to_csv(os.path.join(out, "blind_review_sheet.csv"), index=False)
        m[["id", "source"]].to_csv(os.path.join(out, "blind_review_key.csv"), index=False)
        add(S, "Blind reviewer sheet written", f"{len(m)} items", "2-3 reviewers; detection accuracy < 65%", "TODO",
            "Give blind_review_sheet.csv to reviewers; score against blind_review_key.csv.")

In [ ]:
stage4(te, real, OUT_DIR)
show(scorecard("4"))

### Negative control

If the test is working, splitting the *synthetic* data in half and asking the classifier to tell the halves apart should give an AUC near 0.5. A high AUC here would mean the test itself is broken or the generator is not stationary.

In [ ]:
if RUN_NEGATIVE_CONTROL:
    saved = list(RES); RES.clear()
    half = te.sample(frac=0.5, random_state=0)
    fake_real = pd.DataFrame({"date": half.Date, "timekeeper": half.Timekeeper, "hours": half.Hours,
                              "narrative": half.Narrative, "case_name": "", "firm": ""})
    ctrl_dir = os.path.join(OUT_DIR, "negative_control"); os.makedirs(ctrl_dir, exist_ok=True)
    stage4(te.drop(half.index), fake_real, ctrl_dir)
    ctrl = scorecard("4"); RES[:] = saved
    show(ctrl[ctrl.check.str.startswith("Distinguishability")])

## 7. Stage 5: Utility on the real task (train on synthetic, test on real)

**Goal:** show the synthetic data is useful for what it is meant for.

Compare **TSTR** (train on synthetic, test on real) with **TRTR** (train on real, cross-validated on real) using macro-F1; gate: TSTR is at least 85% of TRTR. Also check whether the ranking of model types agrees between the two.

**Blocker:** matter matching cannot be tested on the fee-application data (no internal matter IDs). What *can* be tested is narrative quality (SPECIFIC / VAGUE / BLOCK_BILLED) if a few hundred real narratives are hand-labeled with the same definitions as the answer key. Provide them via `REAL_LABELS`; otherwise this stage reports NOT RUN.

In [ ]:
def stage5(syn, labels):
    S = "5 Utility"
    if labels is None:
        add(S, "All Stage 5 checks", "NOT RUN", "needs labeled real sample", "NOT RUN",
            "Matter matching cannot be tested on the fee-application data (no internal matter IDs). Label ~150-300 real "
            "narratives as SPECIFIC / VAGUE / BLOCK_BILLED with the same definitions as the synthetic key, then pass --real-labels.")
        return
    L = labels.dropna(subset=["narrative", "quality"]).copy()
    L["quality"] = L.quality.str.upper().str.strip()
    cls = sorted(set(L.quality) & set(syn["K_Narrative Quality"]))
    L = L[L.quality.isin(cls)]
    S_ = syn[syn["K_Narrative Quality"].isin(cls)]
    if len(L) < 60 or L.quality.value_counts().min() < 8:
        add(S, "Labeled real sample", f"{len(L)} rows; {L.quality.value_counts().to_dict()}", ">= 60 rows, >= 8 per class", "WARN", "Underpowered.")
    models = {
        "word LR": lambda: make_pipeline(TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=1), LogisticRegression(max_iter=3000, class_weight="balanced")),
        "char LR": lambda: make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2), LogisticRegression(max_iter=3000, class_weight="balanced")),
    }
    tstr, trtr = {}, {}
    for name, mk in models.items():
        m = mk().fit(S_.Narrative.astype(str), S_["K_Narrative Quality"])
        tstr[name] = f1_score(L.quality, m.predict(L.narrative.astype(str)), average="macro", labels=cls)
        k = int(min(5, L.quality.value_counts().min()))
        if k >= 2:
            p = cross_val_predict(mk(), L.narrative.astype(str), L.quality, cv=StratifiedKFold(k, shuffle=True, random_state=0))
            trtr[name] = f1_score(L.quality, p, average="macro", labels=cls)
    for name in models:
        if name in trtr:
            ratio = tstr[name] / trtr[name] if trtr[name] else float("nan")
            add(S, f"TSTR vs TRTR macro-F1 ({name})", f"TSTR {tstr[name]:.2f} / TRTR {trtr[name]:.2f} = {ratio:.2f}",
                f">= {TSTR_PASS} (warn >= {TSTR_WARN})", "PASS" if ratio >= TSTR_PASS else ("WARN" if ratio >= TSTR_WARN else "FAIL"))
    if len(trtr) == len(models) and len(models) >= 3:
        rho = spearmanr([tstr[k] for k in models], [trtr[k] for k in models])[0]
        add(S, "Model ranking agreement (Spearman)", f"{rho:.2f}", ">= 0.7", "PASS" if rho >= 0.7 else "WARN")
    else:
        add(S, "Model ranking agreement", "needs >= 3 models", "-", "INFO", "Add models to compare rankings on synthetic-trained vs real-trained.")

In [ ]:
labels = read_any(REAL_LABELS) if REAL_LABELS else None
stage5(te, labels)
show(scorecard("5"))

## 8. Stage 6: Privacy and provenance

**Goal:** show nothing was copied from real data and the output is traceable.

- **Provenance:** fingerprints of the workbook and generator, and a source scan confirming the generator reads no external data or network resources.
- **Name scan:** real firm, case and person names (a seed list plus every firm, case and timekeeper in the real file, minus ordinary words) must not appear in the synthetic text or Matter Master.
- **Near-duplicates:** exact matches and the nearest-neighbour similarity from each synthetic narrative to the real ones. Short generic phrases can coincide legitimately, so the gate applies to narratives of six or more words.
- **Manual:** check that synthetic client names (Northgate, Meridian, Tidewater...) do not match real companies.

In [ ]:
SEED_REAL_NAMES = ["Akin Gump", "Purdue", "Celsius", "BlockFi", "FTX", "Sackler", "Genova Burns", "Brown Rudnick",
                   "Kramer Levin", "Otterbourg", "Cole Schotz", "Patterson Belknap", "Sullivan & Cromwell",
                   "King & Spalding", "Bielli", "Klauder", "StakeHound", "LedgerX", "Dietderich", "Dizengoff",
                   "Eckstein", "Molton", "Bouchard", "Hurley", "Lombardi", "Lowenthal"]
GENERIC = {"examine", "documents", "review", "draft", "network", "trading", "pharma", "holdings", "group", "counsel"}


def real_name_terms(real):
    terms = set(SEED_REAL_NAMES)
    common = set()
    if real is not None:
        lc = Counter(w for t in real["narrative"].astype(str) for w in re.findall(r"\b[a-z]{4,}\b", t))
        common = {w for w, n in lc.items() if n >= 5}      # ordinary words, not names
    if real is not None:
        for col in ("firm", "case_name"):
            for v in real[col].dropna().astype(str).unique():
                v = v.strip()
                if v:
                    terms.add(v)
                    first = re.split(r"[\s,]+", v)[0]
                    if len(first) >= 4 and first.lower() not in GENERIC:
                        terms.add(first)
        for v in real["timekeeper"].dropna().astype(str).unique():
            for t in re.findall(r"[A-Za-z]{5,}", v):
                if t.lower() not in GENERIC:
                    terms.add(t)
    return sorted(t for t in terms if t.lower() not in common)


def stage6(syn, mm, real, generator, workbook, selftest=False):
    S = "6 Privacy/provenance"
    sha = hashlib.sha256(open(workbook, "rb").read()).hexdigest()[:16]
    add(S, "Workbook fingerprint (sha256, first 16)", sha, "record", "INFO")
    if generator:
        src = open(generator).read()
        add(S, "Generator source fingerprint (sha256, first 16)", hashlib.sha256(src.encode()).hexdigest()[:16], "record", "INFO")
        pats = r"read_csv|read_excel|read_parquet|read_json|read_sql|\bopen\s*\(|requests\.|urllib|http[s]?://|load_workbook"
        hits = [m.group(0) for m in re.finditer(pats, src)]
        add(S, "Generator reads external data / network", f"{len(hits)} matches {sorted(set(hits)) if hits else ''}", "0", "PASS" if not hits else "FAIL",
            "No input files means no real records can have been copied into the output." if not hits else "Inspect each match.")
    # name scan
    J = lambda col: " ".join(col.fillna("").astype(str))
    corpus = " \n ".join(
        [J(syn.Narrative), J(syn.Client), J(syn.Matter), J(syn.Timekeeper)]
        + [J(mm[c]) for c in ["Client", "Matter", "Codename", "Known Aliases", "Key Terms"] if c in mm]
    )
    terms = real_name_terms(None if selftest else real)
    found = [t for t in terms if len(t) >= 4 and re.search(r"(?<![A-Za-z])" + re.escape(t) + r"(?![A-Za-z])", corpus, re.I)]
    add(S, "Real firm / case / person names found in synthetic text", f"{len(found)} of {len(terms)} terms scanned {found[:10]}", "0",
        "PASS" if not found else "FAIL", "Seed list from the pasted sample" + (" plus names in the real file." if real is not None else "; add --real to scan all firms, cases and timekeepers."))
    add(S, "Synthetic client names vs real companies", "manual check", "no unintended real entities", "TODO",
        "Names such as Northgate, Meridian, Tidewater are generic and may match real companies; web-check before external use.")
    # near-duplicate vs real
    if real is None or selftest:
        add(S, "Synthetic-to-real nearest-neighbour similarity", "NOT RUN", "needs real sample", "NOT RUN" if real is None else "SKIPPED",
            "Selftest uses the same generator on both sides, so overlap is expected." if selftest else "Provide --real.")
        return
    rn = sorted({norm_text(t) for t in real.narrative.astype(str) if len(norm_text(t)) >= 3})
    if len(rn) > 3000:
        import random
        rn = random.Random(0).sample(rn, 3000)   # random, not alphabetical
    sn = sorted({norm_text(t) for t in syn.Narrative.astype(str)})
    exact = sorted(set(rn) & set(sn))
    long_exact = [t for t in exact if len(t.split()) >= 6]
    add(S, "Exact narrative matches synthetic == real", f"{len(exact)} total ({len(exact) - len(long_exact)} short generic, {len(long_exact)} with >= 6 words)",
        "0 with >= 6 words", "PASS" if not long_exact else "FAIL", "Short generic phrases ('update fee tracker') can coincide legitimately.")
    best = []
    sm = SequenceMatcher(autojunk=False)
    for s_ in sn:
        sm.set_seq2(s_)
        b = 0.0
        for r_ in rn:
            sm.set_seq1(r_)
            if sm.real_quick_ratio() > b and sm.quick_ratio() > b:
                v = sm.ratio()
                b = max(b, v)
        best.append((b, s_))
    sims = np.array([b for b, _ in best])
    long_hits = [(b, s_) for b, s_ in best if b >= NN_COPY_SIM and len(s_.split()) >= 6]
    add(S, "Synthetic->real nearest-neighbour char similarity", f"median {np.median(sims):.2f}, 95th pct {np.percentile(sims, 95):.2f}, max {sims.max():.2f}",
        f"no >= {NN_COPY_SIM} with >= 6 words", "PASS" if not long_hits else "FAIL",
        "" if not long_hits else "Examples: " + "; ".join(s_ for _, s_ in long_hits[:3]))

In [ ]:
stage6(te, mm, real, GENERATOR_PY, SYNTHETIC_XLSX)
show(scorecard("6"))

## 9. Combined scorecard and export

In [ ]:
all_scores = pd.DataFrame(RES)
all_scores.to_csv(os.path.join(OUT_DIR, "scorecard_all.csv"), index=False)
counts = all_scores.pivot_table(index="stage", columns="status", values="check", aggfunc="count", fill_value=0)
display(counts)
counts.plot.barh(stacked=True, figsize=(9, 3.6), color={"PASS": "#6cc070", "WARN": "#f2c94c", "FAIL": "#e5534b",
                 "INFO": "#c9ccd1", "NOT RUN": "#9aa0a6", "TODO": "#6aa9ff", "SKIPPED": "#dcdcdc"})
plt.title("Checks by stage and status"); plt.tight_layout(); plt.show()
print("Action items:")
show(all_scores[all_scores.status.isin(["FAIL", "WARN", "TODO", "NOT RUN"])], cols=("status", "check", "value", "note"))

## 10. Findings snapshot

*Snapshot from the v2 run (seed 210, 640 rows; real file 12,000 rows, 11,894 after cleaning). The tables above refresh when the notebook is rerun; this text does not.*

**Stage 1 (integrity).** Keys, input types and hard-case mappings reconcile with the Matter Master. Problems: 12 of 53 FLAG_MISMATCH rows are labelled "correct matter not determinable", which contradicts the Notes; 28 of 53 flags carry no Hard Case tag, including 12 same-client open wrong-matter cases (so the true SAME_CLIENT_WRONG_MATTER count is 29, not 17). Cause: tags are assigned by generator branch rather than from the entered-vs-true relationship, and the determinability guard only covers one branch.

**Stage 2 (split hygiene).** No exact overlap, and TEST narratives are far from TRAIN (nearest-neighbour similarity about 0.55 versus 0.93 within TRAIN). But TEST has only 15 NEEDS_REVIEW, 12 FLAG_MISMATCH and 9 SPLIT_ENTRY rows (gate: 30). Nine TEST split entries reuse a TRAIN template because a split row is labelled TEST if *either* half is held out. Most typed-vague phrases have no held-out variant, so TEST contains only 4 such rows.

**Stage 3 (shortcuts).** A model using only hours, length and separators scores macro-F1 0.25 against 0.22 for the majority class, so there is no broad shortcut; the best baseline (text + entered fields + hours) reaches 0.58, so there is real headroom, mainly on FLAG_MISMATCH. Copying the entered CMID solves 271 of 514 AUTO_MATCH rows (53% of AUTO_MATCH, 42% of all rows), so report the non-trivial subset separately. The one clear lexical tell: "also" and "Separately" appear only in split entries (11 of 16). Length and hours looked like split shortcuts on the whole dataset but vanish within the same narrative quality; they mostly separate multi-part from single-line entries.

**Stage 4 (fidelity).** Real and synthetic are almost perfectly separable (AUC 0.99 on style; 0.94 with domain nouns masked). Casing alone carries most of it: no real entry starts lowercase versus 87% of synthetic. Real entries are longer (median 84 vs 35 chars), end in a period more often (66% vs 14%), use almost no shorthand (about 0% vs 8%), and 41% carry time slices (none synthetic). Hours granularity and the words-hours correlation match. The negative control is near 0.5, so the test itself works.

**Stage 5 (utility).** Not run: needs hand-labeled real narratives.

**Stage 6 (privacy / provenance).** The generator reads no files or network resources; no exact matches with real narratives; closest synthetic-to-real similarity is about 0.93 (gate 0.95; worth inspecting the nearest pair). The single name hit ("Stock") is a false alarm: a real timekeeper surname that is also an ordinary word in synthetic narratives. Client-name collisions remain a manual check.

**Real-data defects found along the way.** The `hours` column is wrong in 799 rows (6.7%): the extractor took the first time slice instead of the total, mostly on Akin Gump block entries (median understatement about 2.3x). The repaired value matches the narrative's slice sum in 787 of 796 checked rows; the original matches none. About 130 rows have a task label in `timekeeper`, about 20% of `firm` values carry a filer prefix, and 106 rows were dropped for slice-sum mismatches (not yet audited).

## 11. Next steps

1. **Generator v2.1:** derive Hard Case tags from the entered-vs-true relationship; drop or relabel the undeterminable flags; draw both halves of a split entry from the same variant pool; remove split-only connectors (or use them in block-billed entries too); parameterize rare-class counts and the TEST rate so every class has at least 30 TEST rows; give every typed-vague phrase a held-out variant.
2. **Build the resolver pipeline** (fit on TRAIN only): CMID normalization, client alias and fuzzy matching, key-term and codename lookup, timekeeper-history candidate filtering, and an action classifier. Report macro-F1 and per-class recall, plus the non-trivial subset; score NEEDS_REVIEW on action only.
3. **Calibrate or replace the noise model** (casing, shorthand, typos) using real *raw* time entries rather than filed fee narratives.
4. **Unblock Stage 5:** hand-label 150-300 real narratives (SPECIFIC / VAGUE / BLOCK_BILLED) or obtain labelled internal entries.
5. **Loose ends:** inspect the closest synthetic-to-real narrative pair (about 0.93) and audit the 106 dropped real rows; fix the `hours` extractor upstream.